In [ ]:
import torch
import optuna
import yaml
from pathlib import Path
import json
import sys
sys.path.append('../')

from src.training.hparam_search import objective
from src.training.trainer import train_model
from src.data.utils import (
    create_features_manifest,
    get_clips_directory_name,
    get_oversample_directory_name,
    extract_dataset_identifier
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

try:
    with open('../config/params.yml', 'r') as f:
        params = yaml.safe_load(f)
except FileNotFoundError:
    print("\nERROR: config/params.yml no encontrado.")
    params = None
except Exception as e:
    print(f"\nERROR al leer config/params.yml: {e}")
    params = None

if not params:
    print("\nNo se pudo cargar la configuración.")
    sys.exit(1)

seed = params.get('random_seed', 42)
exp_params = params.get('experiment', {})
exp_name = exp_params.get('name', 'default_experiment')
exp_base_dir = Path(exp_params.get('base_dir', '../experiments'))
experiment_dir = exp_base_dir / exp_name

print(f"\nCargando configuración para el experimento: {exp_name}")
print(f"Directorio del experimento: {experiment_dir.resolve()}")

# Cargar parámetros
dp_params = params.get('data_pipeline', {})
fe_params = params.get('feature_extraction', {})
training_params = params.get('training', {})
optuna_params = params.get('optuna_search', {})
model_params = params.get('lstm_model', {})
artifact_params = params.get('model_artifacts', {})

# Definir fuente de datos y extraer dataset_id
ds_params = params.get('data_source', {})
raw_videos_dir = Path(ds_params.get('raw_videos_dir', '../data/raw/dataset_videos_original'))
dataset_id = extract_dataset_identifier(raw_videos_dir)

# Parámetros de generación de clips
vp_params = params.get('video_processing', {})
clip_length = vp_params.get('clip_length', 16)
max_segments = vp_params.get('max_segments_per_video', 32)
overlapping = vp_params.get('overlapping', False)
stride = vp_params.get('stride', 8)
balance_mode = dp_params.get('balance_mode', 'none')
max_ratio = vp_params.get('balance_max_ratio', 1.2)

# Generar el nombre del directorio de clips basado en parámetros y dataset
clips_dir_name = get_clips_directory_name(clip_length, max_segments, overlapping, stride, dataset_id)
actual_clips_dir = Path('../data/interim') / clips_dir_name

# Generar directorio de oversample solo si el modo de balanceo lo requiere
if balance_mode == 'oversample':
    oversample_dir_name = get_oversample_directory_name(clip_length, max_segments, overlapping, stride, max_ratio, dataset_id)
    oversample_clips_dir = Path('../data/interim') / oversample_dir_name
else:
    oversample_clips_dir = None  # No se necesita para undersample o none

# Verificar que existan los clips
if not actual_clips_dir.exists():
    print(f"\nERROR: No se encontraron clips en: {actual_clips_dir.name}")
    print(f"  Se esperaban clips generados con los parámetros actuales de params.yml")
    print(f"  Ejecutar primero el notebook 1_data_preparation.ipynb")
    print(f"\nParámetros actuales:")
    print(f"  - dataset_id: {dataset_id}")
    print(f"  - clip_length: {clip_length}")
    print(f"  - max_segments_per_video: {max_segments}")
    print(f"  - overlapping: {overlapping}")
    print(f"  - stride: {stride if overlapping else 'N/A'}")

# Determinar directorio de features basado en el directorio de clips
fe_extractor = fe_params.get('extractor', 'r3d')

# Generar nombre de directorio de features basado en clips
if clips_dir_name.startswith(f"clips_{dataset_id}_"):
    suffix = clips_dir_name.replace(f"clips_{dataset_id}", "")
    features_dir_name = f"features_{dataset_id}{suffix}"
elif clips_dir_name == f"clips_{dataset_id}":
    features_dir_name = f"features_{dataset_id}"
else:
    features_dir_name = clips_dir_name.replace("clips_", "features_")

shared_features_dir = Path('../data/processed') / features_dir_name

# Directorios del experimento
manifests_subdir = experiment_dir / dp_params.get('manifests_subdir', 'results/tables')
models_subdir = experiment_dir / artifact_params.get('models_subdir', 'results/models')

# Crear directorios necesarios
shared_features_dir.mkdir(parents=True, exist_ok=True)
manifests_subdir.mkdir(parents=True, exist_ok=True)
models_subdir.mkdir(parents=True, exist_ok=True)

# Archivos de salida
features_manifest_name = fe_params.get('features_manifest_name', 'manifest_features.csv')
features_manifest_csv = manifests_subdir / features_manifest_name

optuna_results_csv = manifests_subdir / optuna_params.get('results_csv_name', 'optuna_lstm_trials.csv')
final_model_path = models_subdir / artifact_params.get('final_model_name', 'best_lstm_model.pth')
final_metrics_path = manifests_subdir / artifact_params.get('final_metrics_name', 'lstm_final_metrics.json')
training_history_path = manifests_subdir / artifact_params.get('training_history_name', 'lstm_training_history.json')
test_predictions_path = manifests_subdir / artifact_params.get('test_predictions_name', 'lstm_test_predictions.json')

print(f"\n{'='*60}")
print(f"CONFIGURACIÓN AUTOMÁTICA DETECTADA:")
print(f"{'='*60}")
print(f"Dataset identificado: '{dataset_id}'")
print(f"Parámetros de clips:")
print(f"  - clip_length: {clip_length}")
print(f"  - max_segments_per_video: {max_segments}")
print(f"  - overlapping: {overlapping}")
print(f"  - stride: {stride if overlapping else 'N/A'}")
print(f"  - balance_mode: {balance_mode}")
print(f"\nDirectorios generados automáticamente:")
print(f"  Clips base: {actual_clips_dir.resolve()}")
if oversample_clips_dir:
    print(f"  Clips oversample: {oversample_clips_dir.resolve()}")
print(f"  Features: {shared_features_dir.resolve()}")
print(f"  Manifest: {features_manifest_csv.resolve()}")
print(f"{'='*60}\n")

## 1. Extracción de Features y Generación de Manifest

In [ ]:
# Determinar el manifest de clips correcto según el balance_mode
balance_mode = dp_params.get('balance_mode', 'none')
clips_manifest_to_use = None

print(f"Directorio de clips detectado: {actual_clips_dir.name}")

# El nombre base del manifest depende del directorio de clips usado
base_manifest_name = f"manifest_{actual_clips_dir.name}.csv"

if balance_mode == 'oversample':
    # Para oversample, usar el manifest balanceado que incluye clips aumentados
    balanced_manifest = manifests_subdir / "manifest_balanced_oversample.csv"
    if balanced_manifest.exists():
        clips_manifest_to_use = balanced_manifest
        print(f"Modo oversample: usando {balanced_manifest.name}")
    else:
        print(f"ERROR: No se encontró {balanced_manifest.name}")
        print(f"  Ejecutar primero el notebook 1_data_preparation.ipynb con balance_mode='oversample'")
        
elif balance_mode == 'undersample':
    # Para undersample, usar el manifest balanceado que tiene los clips submuestreados
    balanced_manifest = manifests_subdir / "manifest_balanced_undersample.csv"
    if balanced_manifest.exists():
        clips_manifest_to_use = balanced_manifest
        print(f"Modo undersample: usando {balanced_manifest.name}")
    else:
        print(f"ERROR: No se encontró {balanced_manifest.name}")
        print(f"  Ejecutar primero el notebook 1_data_preparation.ipynb con balance_mode='undersample'")
        
else:  # balance_mode == 'none'
    # Sin balanceo, buscar el manifest final o el manifest base correspondiente
    final_manifest = manifests_subdir / "manifest_final.csv"
    base_manifest = manifests_subdir / base_manifest_name
    
    if final_manifest.exists():
        clips_manifest_to_use = final_manifest
        print(f"Modo none: usando {final_manifest.name}")
    elif base_manifest.exists():
        clips_manifest_to_use = base_manifest
        print(f"Modo none: usando {base_manifest.name}")
    else:
        print(f"ERROR: No se encontró manifest de clips")
        print(f"  Se esperaba: {final_manifest.name} o {base_manifest.name}")
        print(f"  Ejecutar primero el notebook 1_data_preparation.ipynb")

# Extraer features desde el manifest correcto
if not clips_manifest_to_use:
    print("\nERROR: No se pudo determinar el manifest de clips a usar")
    print("Ejecutar primero el notebook 1_data_preparation.ipynb")
else:
    print(f"\n{'='*60}")
    print(f"Extrayendo features desde: {clips_manifest_to_use.name}")
    print(f"Clips directory: {actual_clips_dir.name}")
    print(f"Features directory: {shared_features_dir.name}")
    print(f"{'='*60}\n")
    
    # Extraer features
    %run ../src/features/build_features.py \
        --manifest_csv {clips_manifest_to_use} \
        --output_dir {shared_features_dir} \
        --extractor {fe_extractor}
    
    print(f"\nFeatures procesados y guardados en: {shared_features_dir.resolve()}")
    
    # Generar manifest de features para este experimento
    print(f"\n{'='*60}")
    print(f"{'='*60}\n")
    
    create_features_manifest(
        clips_manifest_csv=str(clips_manifest_to_use),
        features_dir=str(shared_features_dir),
        output_csv=str(features_manifest_csv)
    )
    
    print(f"\nManifest de features creado: {features_manifest_csv.resolve()}")

# 2. Entrenamiento de la LSTM

In [ ]:
if not features_manifest_csv.exists():
    print("ERROR: No se encontró el manifest de features.")
    print("Ejecutar primero la sección 1 de este notebook.")
else:
    print(f"Búsqueda de hiperparámetros con Optuna")
    print(f"Usando manifest de features: {features_manifest_csv.name}")
    
    sampler = optuna.samplers.TPESampler(seed=seed)
    study = optuna.create_study(
        direction="minimize", 
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10, interval_steps=1),
        sampler=sampler
    )

    study.optimize(
        lambda trial: objective(
            trial,
            features_manifest=str(features_manifest_csv),
            device=device,
            num_epochs=training_params["epochs"],
            patience=training_params["patience"],
            input_size=model_params.get("input_size", 512),
            seed=seed
        ),
        n_trials=optuna_params["n_trials"],
    )

    print("\nBúsqueda terminada")
    print(f"Mejor trial Loss (validación): {study.best_trial.value:.4f}")
    print("Mejores hiperparámetros:")
    best_params = study.best_trial.params
    print(best_params)
    
    best_trial_auc = study.best_trial.user_attrs.get("best_val_auc", "N/A")
    print(f"AUC de validación del mejor trial: {best_trial_auc:.4f}")

    trials_df = study.trials_dataframe()
    trials_df.to_csv(optuna_results_csv, index=False)

    print(f"\nResultados de todos los trials guardados en: {optuna_results_csv}")


In [ ]:
if not features_manifest_csv.exists():
    print("ERROR: No se encontró el manifest de features.")
    print("Ejecutar primero la sección 1 de este notebook.")
else:
    print(f"Entrenando modelo final con los mejores hiperparámetros")
    print(f"Usando manifest de features: {features_manifest_csv.name}")
    print(f"Usando seed global: {seed}")

    final_config = best_params.copy()
    final_config["input_size"] = model_params.get("input_size", 512)
    final_config["use_attention"] = True

    history, best_val_loss, best_val_auc, best_epoch, test_metrics, test_predictions = train_model(
        config=final_config,
        features_manifest=str(features_manifest_csv),
        save_path=final_model_path,
        device=device,
        num_epochs=training_params["epochs"],
        patience=training_params["patience"],
        seed=seed
    )

    final_results_summary = {
        "best_hyperparameters": best_params,
        "best_validation_loss": best_val_loss,
        "best_validation_auc": best_val_auc,
        "best_epoch": best_epoch,
        "test_metrics": test_metrics,
        "features_manifest": str(features_manifest_csv)
    }

    with open(final_metrics_path, "w") as f:
        json.dump(final_results_summary, f, indent=4)

    with open(training_history_path, "w") as f:
        json.dump(history, f, indent=4)

    with open(test_predictions_path, "w") as f:
        json.dump(test_predictions, f, indent=4)

    print(f"\nMétricas finales del test guardadas en: {final_metrics_path.resolve()}")
    print(f"Modelo final entrenado guardado en: {final_model_path.resolve()}")
    print(f"Historial de entrenamiento guardado en: {training_history_path.resolve()}")
    print(f"Predicciones del test guardadas en: {test_predictions_path.resolve()}")
